# Decision trees – the whole topic in one lab

A decision tree is a sequence of questions such as *"Is the income at most 32 thousand?"*. Each question splits the data in two,
and in each part we keep asking until the groups are "pure". Today we walk the whole way – from a calculation on paper
to the models that win competitions on tabular data:

| part | content |
|---|---|
| 1 | **Impurity and a single split** – Gini, entropy, choosing a threshold by hand |
| 2 | **The best split and a tree step by step** – a function tries every threshold, we keep splitting |
| 3 | **Regression tree** – the same functions, just variance instead of Gini and the mean in a leaf |
| 4 | **A general tree as recursion** – the whole algorithm in one function, comparison with scikit-learn |
| 5 | **Real data** – metrics, train / validation / test data, manual tuning |
| 6 | **Automatic tuning** – cross-validation, grid search, pruning |
| 7 | **Ensembles** – bagging, random forest, gradient boosting |

The functions are **prepared** – you only need to run them. Lines you are meant to change are marked `# EDIT`.
Tasks marked ⭐ are optional – skip them if you are short of time (the separate labs 1–6 in this folder cover them in detail).

In [ ]:
from dataclasses import dataclass      # a shortcut for writing simple classes (we use it for a tree node)

import numpy as np                     # arrays and maths
import pandas as pd                    # tables (DataFrame)
import matplotlib.pyplot as plt        # plots
from IPython.display import display    # nicely formatted tables in the notebook

from sklearn import datasets, metrics  # built-in datasets and evaluation metrics
# tools for splitting data, cross-validation and automatic hyperparameter search
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.dummy import DummyClassifier          # the simplest possible model (baseline)
# decision trees for classification and regression, a text and a graphical printout of a tree
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text, plot_tree
# ensembles = models made of many trees
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, GradientBoostingRegressor,
                              HistGradientBoostingClassifier, AdaBoostClassifier)
from sklearn.linear_model import LogisticRegression  # a simple linear model for comparison
from sklearn.pipeline import make_pipeline           # chains several steps into one model
from sklearn.preprocessing import StandardScaler     # rescales features to mean 0 and standard deviation 1

RSTATE = 42                            # seed of the random generator: everyone gets the same results

# Part 1 – Impurity and a single split

## 1.1 Data

A bank wants to predict from a client's **monthly income** (thousands) and **age** whether they will **repay a loan** (1 = repays, 0 = defaults).
The numbers in the plot are row numbers. You can edit the table at any time and run the notebook again.

In [ ]:
# The table of clients: one row = one client, three columns.
loans = pd.DataFrame(
    [   # income, age, repays
        (18, 25, 0), (22, 45, 0), (25, 60, 0), (30, 22, 0),
        (34, 50, 1), (38, 28, 0), (42, 55, 1), (45, 35, 1),
        (48, 24, 0), (55, 30, 1), (62, 48, 1), (70, 26, 1),
        # EDIT: you can add more rows here
    ],
    columns=["income", "age", "repays"],             # column names
)
CLASS_NAMES = {0: "defaults", 1: "repays"}           # class number -> name shown in the plots
COLORS = {0: "tab:red", 1: "tab:green"}              # class number -> colour of its points


def plot_loans(df, feature=None, threshold=None, ax=None, title=""):
    # Draws the clients in df as points (x = income, y = age) and, optionally, the line of one split.
    if ax is None:                                                          # no axes given -> create a new figure
        _, ax = plt.subplots(figsize=(5.5, 4))
    for cls, group in df.groupby("repays"):                                 # each class in its own colour
        ax.scatter(group["income"], group["age"], s=80, c=COLORS[cls], edgecolor="k", label=CLASS_NAMES[cls])
    for i, row in df.iterrows():                                            # row number next to each point
        ax.annotate(str(i), (row["income"], row["age"]), xytext=(5, 3), textcoords="offset points", fontsize=8)
    if feature == "income":                                                 # split by income -> vertical line
        ax.axvline(threshold, color="k", ls="--", label=f"income <= {threshold}")
    elif feature == "age":                                                  # split by age -> horizontal line
        ax.axhline(threshold, color="k", ls="--", label=f"age <= {threshold}")
    ax.set(xlim=(loans["income"].min() - 5, loans["income"].max() + 5),   # same axes in every plot
           ylim=(loans["age"].min() - 5, loans["age"].max() + 5),
           xlabel="income", ylabel="age", title=title or f"{len(df)} samples")
    ax.legend(fontsize=8)                                                   # legend with class names
    return ax


display(loans.T)                                                            # the table "lying down" (.T = transpose)
plot_loans(loans, title="Bank clients")                                     # all clients, no split yet
plt.show()

**✏️ Task 1.1** – If you could ask **a single** question of the form *"feature ≤ threshold?"*, which one would you choose? Write it down.

## 1.2 Impurity measures

A node is **pure** when it contains samples of a single class. We measure impurity using the class proportions $p_k$ in the node:

| measure | formula | pure node | 2 classes 50 : 50 |
|---|---|---|---|
| **Gini** | $G = 1 - \sum_k p_k^2$ | 0 | 0.5 |
| **Entropy** | $H = -\sum_k p_k \log_2 p_k$ | 0 | 1 |
| **Variance** (regression, part 3) | $\mathrm{Var} = \frac{1}{n}\sum_i (y_i - \bar y)^2$ | 0 | – |

The `CRITERIA` dictionary maps each criterion to a pair of functions: **how to measure impurity** and **what a leaf predicts**
(the most frequent class, or the mean). This is the only difference between a classification and a regression tree – everything else is shared.

In [ ]:
def gini(y):
    # np.unique(..., return_counts=True) returns the distinct classes and how many times each occurs;
    # [1] takes just the counts, dividing by len(y) turns them into proportions p_k
    p = np.unique(y, return_counts=True)[1] / len(y)
    return 1 - np.sum(p ** 2)                            # G = 1 - sum of squared proportions


def entropy(y):
    p = np.unique(y, return_counts=True)[1] / len(y)     # class proportions, as in gini
    return np.sum(p * np.log2(1 / p))                    # H = sum p * log2(1/p), the same as -sum(p * log2(p))


def variance(y):
    return np.mean((y - np.mean(y)) ** 2)                # mean squared deviation from the mean


def majority(y):
    values, counts = np.unique(y, return_counts=True)    # distinct classes and their counts
    return values[np.argmax(counts)]                     # argmax = position of the largest count -> the most frequent class


# For each criterion: (how to measure impurity, what a leaf predicts).
# This dictionary is the ONLY place where classification and regression differ.
CRITERIA = {
    "gini": (gini, majority),
    "entropy": (entropy, majority),
    "variance": (variance, np.mean),
}

# Try the measures on a few made-up nodes with 12 samples each.
for n0, n1 in [(12, 0), (9, 3), (6, 6)]:                 # EDIT: node composition (count of class 0 and 1)
    y_node = np.array([0] * n0 + [1] * n1)               # e.g. (9, 3) -> nine zeros followed by three ones
    print(f"node ({n0:2d} x 0, {n1:2d} x 1):  Gini = {gini(y_node):.3f},  entropy = {entropy(y_node):.3f}")

## 1.3 Quality of a single split

The split *"`feature` ≤ `threshold`?"* sends samples answering **yes** to the left child and the rest to the right child. We judge it by the
**weighted impurity** of the children and the **gain** (decrease in impurity):

$$I_\text{split} = \frac{n_L}{n}\, I(L) + \frac{n_R}{n}\, I(R), \qquad \text{gain} = I(\text{parent}) - I_\text{split}$$

We look for the split with the **largest gain**.

In [ ]:
def split_score(df, target, feature, threshold, criterion="gini"):
    # Scores one split "feature <= threshold" of the table df; returns one row of results as a dictionary.
    impurity = CRITERIA[criterion][0]                    # the impurity function (first item of the pair)
    y = df[target].to_numpy()                            # target values as a NumPy array
    left = (df[feature] <= threshold).to_numpy()         # True/False for every sample: True = it goes left
    n, n_l = len(y), left.sum()                          # samples in the node, samples going left (True counts as 1)
    n_r = n - n_l                                        # samples going right
    imp_l = impurity(y[left]) if n_l else 0.0            # impurity of the left child (an empty child has none)
    imp_r = impurity(y[~left]) if n_r else 0.0           # ~left flips True/False -> the samples going right
    weighted = (n_l * imp_l + n_r * imp_r) / n           # weighted impurity of the children (bigger child counts more)
    return {"feature": feature, "threshold": threshold, "n left": n_l, "n right": n_r,
            "impurity left": imp_l, "impurity right": imp_r,
            "weighted impurity": weighted, "gain": impurity(y) - weighted}   # gain = parent impurity - weighted


FEATURE, THRESHOLD = "income", 40                        # EDIT: your split from task 1.1

display(pd.DataFrame([split_score(loans, "repays", FEATURE, THRESHOLD)]).round(3))   # one row, rounded
plot_loans(loans, FEATURE, THRESHOLD, title=f"Split {FEATURE} <= {THRESHOLD}")       # the split as a dashed line
plt.show()

**✏️ Task 1.2**

* Compute the Gini of the whole table by hand (how many samples of class 0 and 1 are there?) and check: `gini(loans["repays"])`.
* Change `FEATURE` and `THRESHOLD` and look for the largest gain. What is the best gain you found?

# Part 2 – The best split and a tree step by step

There are infinitely many thresholds but only a few **different splits** – it is enough to try thresholds **halfway between neighbouring values**.
The function `all_splits` tries every threshold of every feature and sorts them from the best. The `min_samples_leaf` parameter drops splits
that would leave a child with too few samples (we will use it later).

In [ ]:
def all_splits(df, target, criterion="gini", min_samples_leaf=1):
    # Tries every sensible threshold of every feature and returns a table of splits sorted from the best.
    rows = []                                            # one row (dictionary) per tried split
    for feature in df.columns.drop(target):              # every column except the target
        values = np.unique(df[feature])                  # sorted distinct values of the feature
        # values[:-1] = all but the last, values[1:] = all but the first -> their average = midpoints between neighbours
        for t in (values[:-1] + values[1:]) / 2:
            rows.append(split_score(df, target, feature, t, criterion))
    table = pd.DataFrame(rows)                           # list of dictionaries -> table
    if table.empty:                                      # no threshold at all: the node cannot be split (all samples identical)
        return table
    # keep only splits where both children have at least min_samples_leaf samples
    table = table[(table["n left"] >= min_samples_leaf) & (table["n right"] >= min_samples_leaf)]
    # sort by gain, largest first; "stable" keeps the original order on a tie, so the earlier split wins;
    # reset_index renumbers the rows 0, 1, 2, ... so that row 0 is always the best split
    return table.sort_values("gain", ascending=False, kind="stable").reset_index(drop=True)


splits = all_splits(loans, "repays")                     # all splits of the whole table (the root)
display(splits.head(5).round(3))                         # the five best
print(f"{len(splits)} splits tried, the best: {splits.loc[0, 'feature']} <= {splits.loc[0, 'threshold']}")

Now we build the tree **by hand, step by step**: we split the root with the best split, and for each child that is not pure
we search for the best split again – **using only its own samples**. A leaf predicts the most frequent class of its samples.

In [ ]:
def describe(df, name):
    # Prints a one-line summary of a node: its size, class counts, Gini and whether it is a leaf.
    y = df["repays"]
    print(f"node {name:3s}: n = {len(y):2d}, classes {y.value_counts().sort_index().to_dict()}, "
          f"Gini = {gini(y):.3f}" + ("  -> pure leaf" if gini(y) == 0 else "  -> split further"))


def split(df, feature, threshold):
    # Returns the two parts of the table: rows with feature <= threshold (left) and the rest (right).
    return df[df[feature] <= threshold], df[df[feature] > threshold]


nodes = {"root": loans}                                  # all nodes created so far, by name (we start with the root)


def split_node(name):
    # Splits the node with the given name by its best split and stores both children in `nodes`.
    df = nodes[name]                                     # the samples of this node
    best = all_splits(df, "repays").loc[0]               # row 0 = the best split, found from this node's samples only
    left, right = split(df, best["feature"], best["threshold"])
    prefix = "" if name == "root" else name              # children of the root are "L"/"R", of node "R" they are "RL"/"RR"
    nodes[prefix + "L"], nodes[prefix + "R"] = left, right
    print(f"node {name}: {best['feature']} <= {best['threshold']}")
    describe(left, prefix + "L")
    describe(right, prefix + "R")
    plot_loans(df, best["feature"], best["threshold"], title=f"Node {name} and its best split")
    plt.show()


split_node("root")                                       # step 1: root -> L, R
split_node("R")                                          # step 2: node R -> RL, RR

**✏️ Task 2.1**

* Does the best root split match your guess from task 1.1?
* Node `RL` is not pure yet. Split it in the cell below and continue until all leaves are pure. Draw the tree on paper.
* ⭐ Change the criterion to `"entropy"` (`all_splits(loans, "repays", "entropy")`). Does the order of the splits change?

In [ ]:
split_node("RL")                                         # EDIT: the next node that is not pure

We keep repeating the same thing: *find the best split → split → continue in the children*. That is **recursion** – in part 4 we write it as a function.

# Part 3 – Regression tree

In regression the target is a **number**. The procedure is the same; only two things change:

* the impurity is the **variance** – a split is good when prices in both children are similar;
* a leaf predicts the **mean** of its samples (the mean minimises the sum of squared deviations).

Data: the **age** of a used car (years) and its **price** (thousands).

In [ ]:
# Ten used cars: age in years and price in thousands.
cars = pd.DataFrame(
    [(1, 520), (2, 480), (3, 450), (4, 300), (5, 280),
     (6, 310), (8, 150), (9, 170), (11, 90), (12, 60)],   # EDIT: age, price
    columns=["age", "price"],
)

splits = all_splits(cars, "price", criterion="variance")  # the same function, only a different criterion
display(splits.head(3).round(1))                           # the three best splits

fig, axes = plt.subplots(1, 2, figsize=(12, 4))           # two plots side by side
by_t = splits.sort_values("threshold")                    # the splits ordered by threshold (for a line plot)
axes[0].plot(by_t["threshold"], by_t["gain"], "o-")       # left plot: gain of every threshold
axes[0].set(xlabel="threshold for age", ylabel="gain (decrease in variance)", title="Gain of every threshold")
t = splits.loc[0, "threshold"]                            # threshold of the best split
left, right = split(cars, "age", t)                       # the two leaves after the best split
axes[1].scatter(cars["age"], cars["price"], s=60, edgecolor="k")    # right plot: the cars
# each leaf predicts the mean price of its cars -> two horizontal red lines
axes[1].hlines([left["price"].mean(), right["price"].mean()], [0, t], [t, cars["age"].max() + 1], colors="tab:red", lw=2)
axes[1].axvline(t, color="k", ls="--")                    # the split itself
axes[1].set(xlabel="age", ylabel="price", title=f"Best split age <= {t}: prediction = leaf mean")
for ax in axes:
    ax.grid(alpha=0.3)                                    # light grid in both plots
plt.show()

**✏️ Task 3.1**

* What price does a one-split tree predict for a 7-year-old car? And for a 30-year-old car? (A tree **cannot extrapolate**.)
* ⭐ Change the price of the first car to `2000` (an outlier). How does the best split change? Why does the variance "spend" the whole first split on a single car?

# Part 4 – A general tree as recursion

The whole algorithm fits into one function:

```
build_tree(data):
    create a node with a prediction (most frequent class / mean)
    if the node is pure or the max. depth is reached:   return a leaf
    find the best split (all_splits)
    if no split decreases the impurity:                 return a leaf
    node.left  = build_tree(data where feature <= threshold)
    node.right = build_tree(data where feature >  threshold)
```

`build_tree` uses **the same** `all_splits` and `CRITERIA` functions as the manual calculations above – nothing new is hidden inside.

In [ ]:
@dataclass
class Node:
    # One node of the tree. A leaf has feature = None and no children.
    value: object                  # the node's prediction (class or mean)
    n_samples: int                 # number of training samples
    impurity: float                # impurity of the node
    feature: str = None            # what we split on (None = leaf)
    threshold: float = None        # samples with value <= threshold go left
    left: "Node" = None            # left child (feature <= threshold)
    right: "Node" = None           # right child (feature > threshold)


TRACE = False                                            # EDIT: True = print the building process


def build_tree(df, target, criterion="gini", max_depth=None, min_samples_leaf=1, depth=0):
    # Builds a tree from the table df and returns its root node. depth = depth of the current node (root = 0).
    impurity, leaf_value = CRITERIA[criterion]           # the pair of functions for the chosen criterion
    y = df[target].to_numpy()                            # target values of this node's samples
    node = Node(value=leaf_value(y), n_samples=len(y), impurity=impurity(y))   # every node gets a prediction
    if node.impurity == 0 or (max_depth is not None and depth >= max_depth):
        return node                                      # pure node or maximum depth -> leaf
    splits = all_splits(df, target, criterion, min_samples_leaf)
    if splits.empty or splits.loc[0, "gain"] <= 0:
        return node                                      # no useful split -> leaf
    node.feature, node.threshold = splits.loc[0, "feature"], splits.loc[0, "threshold"]   # remember the best split
    if TRACE:
        print("    " * depth + f"{node.feature} <= {node.threshold}  (n = {node.n_samples})")   # indented by depth
    mask = df[node.feature] <= node.threshold            # True = the sample goes left
    # recursion: the function calls itself on each half of the data, one level deeper
    node.left = build_tree(df[mask], target, criterion, max_depth, min_samples_leaf, depth + 1)
    node.right = build_tree(df[~mask], target, criterion, max_depth, min_samples_leaf, depth + 1)
    return node


def predict(node, df):
    # Returns the tree's prediction for every row of df.
    out = []
    for _, row in df.iterrows():                         # one row (sample) at a time
        n = node                                         # start at the root
        while n.feature is not None:                     # until we reach a leaf, go left / right
            n = n.left if row[n.feature] <= n.threshold else n.right
        out.append(n.value)                              # the leaf's prediction
    return np.array(out)


def print_tree(node, indent=""):
    # Prints the tree as text; every level deeper is indented by "|   ".
    if node.feature is None:                             # a leaf: print its prediction and size
        print(f"{indent}-> {node.value:.4g}   (n = {node.n_samples})")
        return
    print(f"{indent}{node.feature} <= {node.threshold:.4g}")
    print_tree(node.left, indent + "|   ")               # the left subtree (recursion again)
    print(f"{indent}{node.feature} >  {node.threshold:.4g}")
    print_tree(node.right, indent + "|   ")              # the right subtree

Check: our tree and the scikit-learn tree on the data from parts 1 and 3.

In [ ]:
# Classification: our tree on the loans vs. the scikit-learn tree.
root = build_tree(loans, "repays", criterion="gini")
print("=== our classification tree ===")
print_tree(root)
sk = DecisionTreeClassifier(random_state=0).fit(loans[["income", "age"]], loans["repays"])   # features, target
print("=== scikit-learn ===")
print(export_text(sk, feature_names=["income", "age"]))

# Regression: our tree on the cars vs. scikit-learn, both limited to depth 2.
car_tree = build_tree(cars, "price", criterion="variance", max_depth=2)
print("=== our regression tree (max_depth=2) ===")
print_tree(car_tree)
sk_reg = DecisionTreeRegressor(max_depth=2).fit(cars[["age"]], cars["price"])
# np.allclose = are all numbers equal (up to tiny rounding errors)?
print("same predictions as scikit-learn:", np.allclose(predict(car_tree, cars), sk_reg.predict(cars[["age"]])))

**Stopping rules.** An unrestricted tree grows until all leaves are pure – it is then flawless on the training data,
but it also "memorises" the noise. That is why we limit its growth with the hyperparameters `max_depth` (maximum depth) or `min_samples_leaf`
(minimum number of samples in a leaf). The plot shows the regression tree for different depths:

In [ ]:
grid_x = pd.DataFrame({"age": np.linspace(0, 13, 300)})             # 300 evenly spaced ages for drawing the prediction
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)      # four plots side by side, same y axis
for ax, depth in zip(axes, [1, 2, 3, None]):             # EDIT: depths (None = no limit)
    tree = build_tree(cars, "price", "variance", max_depth=depth)    # EDIT: try min_samples_leaf=2
    ax.scatter(cars["age"], cars["price"], s=50, edgecolor="k")      # the cars
    ax.plot(grid_x["age"], predict(tree, grid_x), "tab:red")         # the tree's prediction: a step function
    ax.set(title=f"max_depth = {depth}", xlabel="age")
axes[0].set_ylabel("price")
plt.show()

**✏️ Task 4.1**

* Set `TRACE = True`, run the cell with `Node` and then the check cell again. In what order does the recursion visit the nodes?
* Which of the four regression trees would you choose to predict the price of another car? Why not the deepest one?
* ⭐ **XOR**: run the cell below. Why does our tree not split the data at all, even though two cuts would be enough? What does it say about the **greedy** search for splits?
  Scikit-learn splits even with zero gain, so it solves XOR "by luck" – but add some noise and its first split is still chosen at random.

In [ ]:
# XOR: the class is 1 exactly when x1 and x2 differ.
xor = pd.DataFrame({"x1": [0, 0, 1, 1], "x2": [0, 1, 0, 1], "y": [0, 1, 1, 0]})
print_tree(build_tree(xor, "y"))                         # our tree: stops when no split has a positive gain
display(all_splits(xor, "y")[["feature", "threshold", "gain"]])      # gain of both possible splits
sk_xor = DecisionTreeClassifier(random_state=0).fit(xor[["x1", "x2"]], xor["y"])
print(export_text(sk_xor, feature_names=["x1", "x2"]))     # scikit-learn: splits even with zero gain

# Part 5 – Real data: metrics and manual tuning

Now we use the scikit-learn tree (it does the same as `build_tree`, only faster) on the **Breast Cancer Wisconsin** data:
569 tumours described by 30 features; the goal is to recognise a **malignant** tumour (1).

We split the data into three parts:

| part | share | purpose |
|---|---|---|
| **training** | 60 % | learning the tree |
| **validation** | 20 % | comparing settings – we choose the **hyperparameters** (`max_depth`, `min_samples_leaf`, …) on it |
| **test** | 20 % | the **single** final evaluation – at the end of the lab |

**Metrics** (positive class = malignant):

* **accuracy** – the share of correct predictions;
* **precision** – of the tumours labelled malignant, how many really are malignant;
* **recall** – how many malignant tumours the model found (the most important for a diagnosis: a missed tumour is an expensive mistake);
* **F1** – the harmonic mean of precision and recall; this is what we will optimise.

In [ ]:
cancer = datasets.load_breast_cancer(as_frame=True)      # the data as pandas tables
X = cancer.data                                          # 30 features (columns) of 569 tumours (rows)
y = (cancer.target == 0).astype(int)                     # in scikit-learn 0 = malignant; we want 1 = malignant (positive class)

# First split off 20 % for the test, then split the rest into training (60 %) and validation (20 %).
# stratify keeps the same share of malignant tumours in every part.
X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RSTATE)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.25, stratify=y_trval,
                                                  random_state=RSTATE)        # 0.25 * 80 % = 20 %
print(f"training: {len(y_train)}, validation: {len(y_val)}, test: {len(y_test)} samples, "
      f"share of malignant: {y.mean():.2f}")


def scores(y_true, y_pred):
    # The four metrics of a classifier: true classes vs. predicted classes.
    return {"accuracy": metrics.accuracy_score(y_true, y_pred),
            "precision": metrics.precision_score(y_true, y_pred, zero_division=0),   # 0 if nothing was predicted positive
            "recall": metrics.recall_score(y_true, y_pred),
            "F1": metrics.f1_score(y_true, y_pred)}


baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)   # always "benign"
full_tree = DecisionTreeClassifier(random_state=RSTATE).fit(X_train, y_train)  # unrestricted tree
display(pd.DataFrame({                                   # one column per model / data part, .T turns them into rows
    "baseline (validation)": scores(y_val, baseline.predict(X_val)),
    "tree (training)": scores(y_train, full_tree.predict(X_train)),
    "tree (validation)": scores(y_val, full_tree.predict(X_val)),
}).T.round(3))
print(f"unrestricted tree: depth {full_tree.get_depth()}, {full_tree.get_n_leaves()} leaves")
# confusion matrix: rows = true class, columns = predicted class
metrics.ConfusionMatrixDisplay.from_predictions(y_val, full_tree.predict(X_val),
                                                display_labels=["benign", "malignant"], cmap="Blues")
plt.title("Confusion matrix – validation data")
plt.show()

The baseline has decent accuracy but zero recall – that is why accuracy alone is not enough with imbalanced classes.
The unrestricted tree is flawless on the training data and worse on the validation data: **overfitting**.

## 5.1 Manual hyperparameter tuning

Add settings to the `ATTEMPTS` list and watch the **validation F1**. The table is sorted from the best attempt.

| hyperparameter | what it does |
|---|---|
| `max_depth` | maximum depth of the tree |
| `min_samples_leaf` | minimum number of samples in a leaf |
| `min_samples_split` | minimum number of samples a node needs to be split |
| `max_leaf_nodes` | maximum number of leaves |
| `criterion` | `"gini"` or `"entropy"` |
| `class_weight` | `"balanced"` = mistakes on the smaller class weigh more |

In [ ]:
ATTEMPTS = [                                             # EDIT: add more attempts
    {},                                                  # {} = default settings (unrestricted tree)
    {"max_depth": 3},
    {"max_depth": 4, "min_samples_leaf": 5},
]

log = []                                                 # one row per attempt
for params in ATTEMPTS:
    # **params unpacks the dictionary into arguments: {"max_depth": 3} -> max_depth=3
    tree = DecisionTreeClassifier(random_state=RSTATE, **params).fit(X_train, y_train)
    row = {"hyperparameters": str(params), "depth": tree.get_depth(), "leaves": tree.get_n_leaves(),
           "train F1": metrics.f1_score(y_train, tree.predict(X_train))}
    row.update({f"val. {k}": v for k, v in scores(y_val, tree.predict(X_val)).items()})   # add validation metrics
    log.append(row)
display(pd.DataFrame(log).sort_values("val. F1", ascending=False).round(3))   # the best attempt first

The plot shows the effect of one hyperparameter: the training F1 grows with the complexity of the tree, the validation F1 drops from some point on.

In [ ]:
PARAM, VALUES = "max_depth", [1, 2, 3, 4, 5, 6, 8, 10, None]   # EDIT: e.g. "min_samples_leaf", [1, 2, 5, 10, 20, 40]

train_f1, val_f1 = [], []                                # F1 for every value, on training and on validation data
for v in VALUES:
    tree = DecisionTreeClassifier(random_state=RSTATE, **{PARAM: v}).fit(X_train, y_train)   # e.g. max_depth=v
    train_f1.append(metrics.f1_score(y_train, tree.predict(X_train)))
    val_f1.append(metrics.f1_score(y_val, tree.predict(X_val)))
plt.figure(figsize=(7, 3.5))
plt.plot([str(v) for v in VALUES], train_f1, "o-", label="training")   # str() so that None can be an axis label
plt.plot([str(v) for v in VALUES], val_f1, "s-", label="validation")
plt.xlabel(PARAM); plt.ylabel("F1"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

**✏️ Task 5.1** – Find by hand a setting with the best validation F1 you can (at least 5 attempts). Write your best setting into `MANUAL`:

In [ ]:
MANUAL = {"max_depth": 4, "min_samples_leaf": 5}         # EDIT: your best setting

tree = DecisionTreeClassifier(random_state=RSTATE, **MANUAL).fit(X_train, y_train)
# feature_importances_: one number per feature, together they sum to 1
importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Most important features (the decrease in impurity brought by splits on the feature):")
display(importance[importance > 0].round(3).to_frame("importance"))   # only features the tree actually uses

plt.figure(figsize=(14, 5))
# draw only the top two levels of the tree (max_depth=2 here limits the drawing, not the tree)
plot_tree(tree, max_depth=2, feature_names=list(X.columns), class_names=["benign", "malignant"],
          filled=True, rounded=True, fontsize=8)
plt.title("Top of the tree")
plt.show()

**✏️ Task 5.2** – ⭐ Change the `random_state` of the data split (`RSTATE` in the first cell) and run part 5 again.
Does the best setting and the most important feature stay the same? Trees are **unstable**: a small change of the data can change the whole tree.
A single validation set (114 samples) is also rather small – choosing by it is partly random. The next two parts deal with both.

# Part 6 – Automatic tuning

## 6.1 Cross-validation

Instead of a single validation set we split the training + validation data (80 %) into $k$ parts (**folds**). We train the model $k$ times,
each time on $k-1$ parts, and evaluate it on the remaining one. The result is a mean **and** a spread of the estimate – each sample is used for validation once.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RSTATE)    # stratification = same class shares in every fold
# trains and evaluates the tree 5 times (each fold is the validation part once) and returns 5 F1 values
f1 = cross_val_score(DecisionTreeClassifier(random_state=RSTATE, **MANUAL), X_trval, y_trval, cv=cv, scoring="f1")
print("F1 in the individual folds:", f1.round(3))
print(f"mean {f1.mean():.3f} ± {f1.std():.3f}")          # mean ± standard deviation

## 6.2 Grid search

`GridSearchCV` tries **every combination** of values in the grid, scores each by cross-validation and picks the best.
It is exactly our manual procedure from part 5 – only systematic and with a better estimate of quality.

In [ ]:
param_grid = {                                           # EDIT: the grid of values
    "max_depth": [2, 3, 4, 5, 6, 8, None],
    "min_samples_leaf": [1, 2, 5, 10, 20],
    "criterion": ["gini", "entropy"],
}
grid = GridSearchCV(DecisionTreeClassifier(random_state=RSTATE), param_grid, cv=cv, scoring="f1")
grid.fit(X_trval, y_trval)                               # tries every combination with cross-validation
n = len(grid.cv_results_["params"])                      # number of tried combinations
print(f"{n} combinations x 5 folds = {5 * n} trained trees")
print("best hyperparameters:", grid.best_params_)
print(f"best cross-validated F1: {grid.best_score_:.3f}")

# Heat map: mean CV F1 for every max_depth x min_samples_leaf (only the best criterion).
res = pd.DataFrame(grid.cv_results_)                     # all results as a table
res = res[res["param_criterion"] == grid.best_params_["criterion"]]
# rows = max_depth (as text, so that None works), columns = min_samples_leaf, values = mean CV F1
heat = res.pivot_table(index=res["params"].apply(lambda p: str(p["max_depth"])),
                       columns="param_min_samples_leaf", values="mean_test_score", sort=False)
fig, ax = plt.subplots(figsize=(6.5, 4.5))
im = ax.imshow(heat.values, cmap="viridis")              # the table as coloured squares
ax.set(xticks=range(heat.shape[1]), xticklabels=heat.columns, yticks=range(heat.shape[0]), yticklabels=heat.index,
       xlabel="min_samples_leaf", ylabel="max_depth", title=f"CV F1 (criterion = {grid.best_params_['criterion']})")
for i in range(heat.shape[0]):                           # write the value into every square
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{heat.values[i, j]:.3f}", ha="center", va="center", color="w", fontsize=8)
plt.colorbar(im, ax=ax)                                  # colour scale on the side
plt.show()

With large grids the number of combinations explodes – then `RandomizedSearchCV` is used, which tries only a given number of
**random** combinations (often just as good in a fraction of the time).

## 6.3 Cost complexity pruning

A different approach: we let the tree grow fully and then **prune** it. The parameter `ccp_alpha` is a "price per leaf" – the tree minimises
$\text{error} + \alpha \cdot \text{number of leaves}$. A larger $\alpha$ = a smaller tree. Scikit-learn computes all values of $\alpha$
at which the tree shrinks; we pick the best one by cross-validation.

In [ ]:
# All values of ccp_alpha at which the fully grown tree loses some leaves.
path = DecisionTreeClassifier(random_state=RSTATE).cost_complexity_pruning_path(X_trval, y_trval)
rows = []
for alpha in path.ccp_alphas[:-1]:                       # the last alpha prunes the tree down to the root
    t = DecisionTreeClassifier(ccp_alpha=alpha, random_state=RSTATE)
    rows.append({"ccp_alpha": alpha, "leaves": t.fit(X_trval, y_trval).get_n_leaves(),   # size of the pruned tree
                 "CV F1": cross_val_score(t, X_trval, y_trval, cv=cv, scoring="f1").mean()})
pruning = pd.DataFrame(rows)
# rows with the highest CV F1; .iloc[-1] takes the last of them = the largest alpha = the smallest tree
best_alpha = float(pruning.loc[pruning["CV F1"] == pruning["CV F1"].max(), "ccp_alpha"].iloc[-1])

plt.figure(figsize=(7, 3.5))
plt.plot(pruning["leaves"], pruning["CV F1"], "o-")
plt.xlabel("number of leaves of the pruned tree"); plt.ylabel("CV F1"); plt.grid(alpha=0.3)
plt.title(f"Pruning: best ccp_alpha = {best_alpha:.4f}")
plt.show()

## 6.4 Comparison on the test data

Only now – after the hyperparameters have been chosen – do we use the test data. We train each model on the training + validation data
and evaluate it **once**.

In [ ]:
candidates = {                                           # method name -> hyperparameters
    "unrestricted tree": {},
    "manual tuning (part 5)": MANUAL,
    "grid search": grid.best_params_,
    "pruning": {"ccp_alpha": best_alpha},
}
rows = []
for name, params in candidates.items():
    t = DecisionTreeClassifier(random_state=RSTATE, **params).fit(X_trval, y_trval)   # learn on training + validation
    rows.append({"method": name, "leaves": t.get_n_leaves(),
                 "CV F1": cross_val_score(t, X_trval, y_trval, cv=cv, scoring="f1").mean(),
                 # ** merges the test metrics into the row, with "test " in front of each name
                 **{f"test {k}": v for k, v in scores(y_test, t.predict(X_test)).items()}})
display(pd.DataFrame(rows).round(3))

**✏️ Task 6.1**

* Did the model with the best CV F1 win on the test? The test set has only 114 samples – one tumour more or less
  moves the F1 by roughly 0.02. Differences smaller than that are not conclusive.
* Why must we not choose hyperparameters by the test data?
* ⭐ Add `"class_weight": [None, "balanced"]` to the grid. Does the recall improve?

# Part 7 – Ensembles: many trees are better than one

A single deep tree has a small **bias** (it can describe a complex relationship) but a large **variance** (any change of the data changes it a lot).
Ensembles combine many trees:

| method | idea | what it improves |
|---|---|---|
| **bagging** | many deep trees on random samples of the data, we **average** the results | variance |
| **random forest** | bagging + only a random subset of features at each split (trees are less alike) | variance |
| **boosting** | small trees **one after another**, each fixes the errors of the previous ones | bias |

## 7.1 Bagging by hand

**Bootstrap** = drawing $n$ samples out of $n$ **with replacement** (some repeat, about a third are missing). We train a deep tree on each sample
and average the predictions. Data: a noisy sine.

In [ ]:
rng = np.random.default_rng(0)                           # random number generator with a fixed seed
def make_sine(n):
    # n random points x in [0, 6] and y = sin(x) + noise
    x = rng.uniform(0, 6, size=n)
    return x.reshape(-1, 1), np.sin(x) + rng.normal(0, 0.3, size=n)   # reshape: scikit-learn wants a column (matrix)

X_tr, y_tr = make_sine(80)                               # training data
X_te, y_te = make_sine(500)                              # test data
xs = np.linspace(0, 6, 500).reshape(-1, 1)               # evenly spaced points for drawing the curves


def bootstrap(X, y, rng):
    # A bootstrap sample: n samples drawn out of n with replacement.
    idx = rng.integers(0, len(y), size=len(y))           # random indices with replacement (some repeat)
    return X[idx], y[idx]


def bagging_predict(X_train, y_train, X_new, n_trees, seed=0):
    # Trains n_trees deep trees, each on its own bootstrap sample, and averages their predictions for X_new.
    rng = np.random.default_rng(seed)
    predictions = []                                     # predictions of every tree
    for _ in range(n_trees):
        Xb, yb = bootstrap(X_train, y_train, rng)
        predictions.append(DecisionTreeRegressor().fit(Xb, yb).predict(X_new))   # an unrestricted (deep) tree
    return np.mean(predictions, axis=0)                  # average over all trees


fig, axes = plt.subplots(1, 3, figsize=(15, 3.6), sharey=True)
for ax, n_trees in zip(axes, [1, 10, 200]):              # EDIT: numbers of trees
    mse = np.mean((bagging_predict(X_tr, y_tr, X_te, n_trees) - y_te) ** 2)   # test mean squared error
    ax.scatter(X_tr, y_tr, s=12)                         # training points
    ax.plot(xs, np.sin(xs), "k--", lw=1)                 # the true curve without noise
    ax.plot(xs, bagging_predict(X_tr, y_tr, xs, n_trees), "tab:red")   # the bagging prediction
    ax.set_title(f"{n_trees} trees, test MSE {mse:.3f}  (noise: {0.3 ** 2:.2f})")   # noise variance = the lowest possible MSE
plt.show()

## 7.2 Gradient boosting by hand

Boosting starts with a constant prediction (the mean) and then repeats: compute the **residuals** (what the model has not explained yet), train
a **small** tree on them and add its prediction multiplied by the **learning rate**. It is just a few lines:

In [ ]:
def gradient_boosting(X, y, n_trees=100, learning_rate=0.1, max_depth=2):
    # Returns the starting value f0 and the list of trees; the model is f0 + learning_rate * (sum of tree predictions).
    f0 = y.mean()                                        # start: a constant prediction
    pred = np.full(len(y), f0)                           # current prediction for every training sample
    trees = []
    for _ in range(n_trees):
        residuals = y - pred                             # what the model has not explained yet
        tree = DecisionTreeRegressor(max_depth=max_depth, random_state=0).fit(X, residuals)   # a small tree learns the residuals
        pred += learning_rate * tree.predict(X)          # a small step towards the residuals
        trees.append(tree)
    return f0, trees


def boosting_predict(f0, trees, X, learning_rate=0.1):
    # Prediction of the boosted model made of the given trees.
    return f0 + learning_rate * sum(t.predict(X) for t in trees)


LR = 0.1                                                 # EDIT: learning rate
f0, trees = gradient_boosting(X_tr, y_tr, n_trees=300, learning_rate=LR)

fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharey=True)
for ax, m in zip(axes, [1, 10, 50, 300]):                # EDIT: numbers of trees shown
    # trees[:m] = only the first m trees -> how the model looked after m steps
    mse = np.mean((boosting_predict(f0, trees[:m], X_te, LR) - y_te) ** 2)
    ax.scatter(X_tr, y_tr, s=12)
    ax.plot(xs, np.sin(xs), "k--", lw=1)
    ax.plot(xs, boosting_predict(f0, trees[:m], xs, LR), "tab:red")
    ax.set_title(f"after {m} trees, test MSE {mse:.3f}")
plt.show()

# The same model from scikit-learn: the predictions should be equal up to rounding errors.
sk_gb = GradientBoostingRegressor(n_estimators=300, learning_rate=LR, max_depth=2, random_state=0).fit(X_tr, y_tr)
print(f"largest difference from scikit-learn: {np.abs(boosting_predict(f0, trees, X_te, LR) - sk_gb.predict(X_te)).max():.1e}")

**✏️ Task 7.1**

* Why does bagging use **deep** trees and boosting **shallow** ones?
* Set `LR = 1.0` and `LR = 0.01`. What happens after 10 and after 300 trees? (A smaller learning rate = more trees needed,
  but usually a better result. Too many trees make boosting overfit.)

## 7.3 Comparison on real data

Back to the tumours: we compare all models by cross-validation on the same data as in part 6. For comparison we also add
a simple **logistic regression** – trees are not always the best choice.

In [ ]:
models = {                                               # EDIT: add / change models
    "unrestricted tree": DecisionTreeClassifier(random_state=RSTATE),
    "tuned tree (grid search)": DecisionTreeClassifier(random_state=RSTATE, **grid.best_params_),
    "random forest (300 trees)": RandomForestClassifier(n_estimators=300, random_state=RSTATE),
    "AdaBoost": AdaBoostClassifier(n_estimators=200, random_state=RSTATE),
    "gradient boosting": GradientBoostingClassifier(random_state=RSTATE),
    "hist. gradient boosting": HistGradientBoostingClassifier(random_state=RSTATE),
    # logistic regression needs features on a similar scale -> rescale first, inside the pipeline
    "logistic regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
}
rows = []
for name, model in models.items():
    f1 = cross_val_score(model, X_trval, y_trval, cv=cv, scoring="f1")   # 5-fold CV, the same folds as in part 6
    rows.append({"model": name, "CV F1": f1.mean(), "std": f1.std()})
display(pd.DataFrame(rows).sort_values("CV F1", ascending=False).round(3))   # the best model first

# oob_score=True: each sample is evaluated by the trees that did not see it during training
forest = RandomForestClassifier(n_estimators=300, oob_score=True, random_state=RSTATE).fit(X_trval, y_trval)
print(f"random forest – OOB accuracy (an estimate without validation data): {forest.oob_score_:.3f}")

**OOB (out-of-bag)**: each tree of the forest has not seen about a third of the samples (they were not in its bootstrap sample). We can evaluate it on them –
an estimate of quality "for free", without cross-validation.

**Other options** you will meet in practice:

* **XGBoost, LightGBM, CatBoost** – fast, refined gradient boosting libraries (regularisation, missing values,
  categorical features); on tabular data they are among the strongest models. `HistGradientBoostingClassifier` is their counterpart in scikit-learn.
* **Extra Trees** – a random forest that also chooses thresholds at random.
* **Voting / stacking** – combining different models (e.g. a forest and logistic regression) by averaging or by another model.

## 7.4 Final test

**✏️ Task 7.2** – Choose one model by CV F1, train it on the training + validation data and evaluate it on the test **once**.

In [ ]:
FINAL = RandomForestClassifier(n_estimators=300, random_state=RSTATE)   # EDIT: your chosen model

FINAL.fit(X_trval, y_trval)                              # learn on training + validation data
pred = FINAL.predict(X_test)                             # the one and only use of the test data
display(pd.DataFrame([scores(y_test, pred)]).round(3))
metrics.ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["benign", "malignant"], cmap="Blues")
plt.show()

# Summary

* A tree repeatedly splits the data by questions *"feature ≤ threshold?"*; at each node it picks the split with the largest **decrease in impurity** (greedily).
* **Classification**: Gini / entropy, leaf = most frequent class. **Regression**: variance, leaf = mean. Otherwise the algorithm is the same.
* An unrestricted tree **overfits**. We limit its growth (`max_depth`, `min_samples_leaf`, …) or **prune** it (`ccp_alpha`).
* Hyperparameters are chosen on validation data or by **cross-validation** (`GridSearchCV`); the test is used **once**, at the end.
* A single tree is readable but unstable. A **random forest** (an average of many trees) reduces variance, **boosting** (successive corrections) reduces bias.
* Always compare with a baseline and a simple model – sometimes logistic regression wins.

# Homework (⭐)

1. **Regression on real data** – repeat parts 5 and 6 for `datasets.load_diabetes()` with `DecisionTreeRegressor`
   and the metrics MAE, RMSE and $R^2$. Compare with `LinearRegression`. (In detail: labs 4 and 5.)
2. **A new criterion** – add a `"mae"` criterion to `CRITERIA`: impurity = mean absolute deviation from the median, leaf = median.
   Build a tree with it on `cars` with the outlier from task 3.1. (In detail: labs 2 and 3.)
3. **Tuning a forest** – use `GridSearchCV` to tune `max_features` and `min_samples_leaf` of `RandomForestClassifier`. (In detail: lab 6.)